# 03 — Modelos: comparação na validação

Treina os modelos no período de treino e compara no de validação (separação feita em `02_separacao.ipynb`). O período de **teste não é carregado aqui**: ele fica guardado para o resultado final, na Etapa 4.

In [1]:
import numpy as np
import pandas as pd
from sklearn.metrics import average_precision_score, roc_auc_score

treino = pd.read_parquet("../data/processed/treino.parquet")
validacao = pd.read_parquet("../data/processed/validacao.parquet")

alvo = "is_canceled"
variaveis = treino.columns.drop([alvo, "arrival_date"])
categoricas = treino[variaveis].select_dtypes(exclude="number").columns.tolist()
numericas = treino[variaveis].select_dtypes(include="number").columns.tolist()

X_treino, y_treino = treino[variaveis], treino[alvo]
X_validacao, y_validacao = validacao[variaveis], validacao[alvo]
print(f"{len(categoricas)} categóricas, {len(numericas)} numéricas")

10 categóricas, 12 numéricas


## 3.2 Métricas e pontos de comparação

Cada modelo dá uma nota de risco (probabilidade de cancelar) para cada reserva. Três métricas:
- **ROC-AUC:** a chance de o modelo dar nota maior a uma reserva que cancela do que a uma que não cancela. 0,5 é sorteio; 1 é perfeito.
- **PR-AUC:** mede o acerto olhando só para as canceladas. O ponto de partida é a própria taxa de cancelamento (28% na validação), então é mais exigente quando a classe que interessa é minoria.
- **Cancelam no top 10%:** das 10% reservas com maior nota, quantas cancelaram de fato. É a métrica do uso: se o hotel só consegue ligar para 10% das reservas, quantas ligações caem numa reserva que ia cancelar.

In [2]:
def taxa_no_topo(y, prob, fracao=0.10):
    """Entre as reservas com maior nota (as 10% mais arriscadas), a fração que cancelou.
    Reservas empatadas no corte entram pela média delas."""
    y, prob = np.asarray(y), np.asarray(prob)
    k = int(len(prob) * fracao)
    corte = np.sort(prob)[-k]
    acima, empate = prob > corte, prob == corte
    return (y[acima].sum() + (k - acima.sum()) * y[empate].mean()) / k


resultados = []


def avaliar(nome, prob):
    """Calcula as três métricas na validação, guarda e devolve a tabela com todos os modelos até agora."""
    resultados.append({
        "modelo": nome,
        "ROC-AUC": roc_auc_score(y_validacao, prob),
        "PR-AUC": average_precision_score(y_validacao, prob),
        "cancelam no top 10%": taxa_no_topo(y_validacao, prob),
    })
    return pd.DataFrame(resultados).set_index("modelo").round(3)

**Pontos de comparação, antes de qualquer modelo:**
1. **Taxa média** — a mesma nota para todas as reservas. É o equivalente a sortear.
2. **Regra da antecedência** — a nota é o próprio `lead_time`: quanto mais cedo a reserva, mais risco.
3. **Tabela segmento × antecedência** — a taxa de cancelamento do treino em cada combinação de segmento e faixa de antecedência. É o que um analista montaria numa planilha.

Um modelo só vale a pena se superar a tabela com folga.

In [3]:
avaliar("taxa média", np.full(len(y_validacao), y_treino.mean()))
avaliar("regra: antecedência", X_validacao["lead_time"])

faixas = [-1, 7, 30, 90, 180, 365, np.inf]


def grupo(X):
    return X["market_segment"].astype(str) + " | " + pd.cut(X["lead_time"], faixas).astype(str)


taxa_por_grupo = y_treino.groupby(grupo(X_treino)).mean()
avaliar("tabela: segmento × antecedência", grupo(X_validacao).map(taxa_por_grupo).fillna(y_treino.mean()))

,ROC-AUC,PR-AUC,cancelam no top 10%
modelo,,,
taxa média,0.500,0.284,0.284
regra: antecedência,0.682,0.430,0.509
tabela: segmento × antecedência,0.735,0.466,0.556


Sorteando, 28% das ligações cairiam numa reserva que ia cancelar. Só com a antecedência isso sobe para 51%, e a tabela de segmento × antecedência chega a **56%**. Essa é a régua para os modelos.